# Stage 2 — GRPO Self-Play in Pure Sim

**Goal:** Train Llama-3.2-1B (post-SFT) on the 3-agent negotiation env using **real GRPO**: Monte-Carlo policy gradient with group-relative advantages, KL penalty against frozen SFT reference, PPO-style clip ratio, and entropy regularization.

**Runtime:** ~3.5 hours on A10G small.
**Prereq:** SFT checkpoint at `Jayyyy234/agentgrid-sft`.

**Algorithm components actually implemented:**
- Group sampling: `G=4` rollouts per seed → group-relative advantage `A = (R - mean) / (std + 1e-8)`
- PPO-clip surrogate (ε = 0.2)
- KL penalty β = 0.04 against frozen reference (SFT, accessed via `disable_adapter` context)
- Entropy bonus α = 0.01
- AdamW lr 5e-6, linear warmup 5%, grad-clip 1.0
- Loss masked to completion tokens only (prompt tokens have weight 0)
- Curriculum scaled across run: easy → medium → full
- Adapter checkpoints every 5 groups for resume safety

Run cells **in order**, do not skip Cell 1 (env-var must be set before any `agentgrid_env` import).

In [ ]:
# Trust-decision audit — single-file append across all episodes
# Must be set BEFORE importing agentgrid_env (constant captured at import time)
import os
TRUST_DECISIONS_PATH = '/tmp/workspace/trust_decisions.jsonl'
os.environ['AGENTGRID_TRUST_DECISIONS_PATH'] = TRUST_DECISIONS_PATH
# Wipe stale data from previous runs in this notebook session
if os.path.exists(TRUST_DECISIONS_PATH):
    os.remove(TRUST_DECISIONS_PATH)
print(f'Trust decisions will append to: {TRUST_DECISIONS_PATH}')

In [ ]:
import os
from huggingface_hub import login

hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    login(token=hf_token)
    print("Logged in via HF_TOKEN secret.")
else:
    login()  # fallback: interactive prompt
    print("Logged in interactively.")

In [ ]:
import importlib, subprocess, sys

def _pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

_missing = [p for p in ("transformers", "peft", "bitsandbytes", "accelerate", "trl", "datasets")
            if importlib.util.find_spec(p) is None]
if _missing:
    print(f"Installing: {_missing}")
    _pip("torch==2.5.1+cu118", "torchvision==0.20.1+cu118",
         "--index-url", "https://download.pytorch.org/whl/cu118")
    _pip("transformers", "peft", "accelerate", "bitsandbytes", "trl",
         "datasets", "plotly", "matplotlib", "pandas")
    print("Done.")
else:
    print("All dependencies present.")


In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_DIR = Path("/tmp/workspace/AgentGrid_V1")

if not REPO_DIR.exists():
    print("Workspace missing — cloning repo...")
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    subprocess.check_call(["git", "clone",
        "https://github.com/jayyyyqwq/GaN_J_AI.git", str(REPO_DIR)])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)])
    print("Repo cloned and installed.")

os.chdir(str(REPO_DIR))
print("Working dir:", os.getcwd())


In [ ]:
import json, os, random, time
from pathlib import Path

# ── Hub repos ───────────────────────────────────────────────────────
SFT_CHECKPOINT = "Jayyyy234/agentgrid-sft"
GRPO_REPO      = "Jayyyy234/agentgrid-grpo"

# ── Env shape ───────────────────────────────────────────────────────
AGENTS         = ["A", "B", "C"]
EPISODE_STEPS  = 50

# ── GRPO budget ─────────────────────────────────────────────────────
# Quick sanity run: N_GROUPS=5  (~20 min on A100,  20 episodes)
# Full training:   N_GROUPS=25  (~3.5 h on A10G, 100 episodes)
N_GROUPS       = int(os.environ.get("N_GROUPS", 5))
G              = 4
N_EPISODES     = N_GROUPS * G
CURRICULUM_SCALE = 500 / N_EPISODES

# ── GRPO hyperparameters ─────────────────────────────────────────────
GRPO_LR        = 5e-7
WARMUP_FRAC    = 0.05
CLIP_EPS       = 0.1
KL_BETA        = 0.1
ENT_ALPHA      = 0.01
GRAD_CLIP      = 1.0
TEMP_START     = 0.9
TEMP_END       = 0.7

# ── Sampling / model ────────────────────────────────────────────────
MAX_NEW_TOKENS = 80
MAX_SEQ_LEN    = 2048
LORA_R         = 16

# ── Logging / checkpoints ───────────────────────────────────────────
OUTPUT_DIR     = "/tmp/workspace/grpo_output"
CKPT_DIR       = "/tmp/workspace/grpo_ckpts"
LOG_PATH       = "/tmp/workspace/grpo_rewards.jsonl"
METRICS_PATH   = "/tmp/workspace/grpo_metrics.jsonl"
SAMPLES_PATH   = "/tmp/workspace/grpo_samples.txt"
CKPT_EVERY     = 5

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)
for p in [LOG_PATH, METRICS_PATH, SAMPLES_PATH]:
    if os.path.exists(p):
        os.remove(p)

print(f"GRPO config: {N_GROUPS} groups × G={G} = {N_EPISODES} episodes, "
      f"lr={GRPO_LR}, β={KL_BETA}, α={ENT_ALPHA}, clip={CLIP_EPS}")
print(f"Tip: set env var N_GROUPS=25 before running for the full training run.")


## Step 1: Load policy + reference

The policy is the SFT checkpoint with new trainable LoRA adapters on top.
The reference is the same model with adapters disabled (via `peft.disable_adapter()`) — saves ~1 GB VRAM vs loading SFT twice.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel, LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

BASE_MODEL = "meta-llama/Llama-3.2-1B"
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
)
base = prepare_model_for_kbit_training(base, use_gradient_checkpointing=True)

try:
    tokenizer = AutoTokenizer.from_pretrained(SFT_CHECKPOINT)
    model = PeftModel.from_pretrained(base, SFT_CHECKPOINT, is_trainable=True)
    print(f"Loaded SFT adapter from {SFT_CHECKPOINT}")
except Exception as e:
    print(f"SFT checkpoint unavailable ({e}).")
    print("Attaching a fresh LoRA adapter to the base model instead.")
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
    lora_cfg = LoraConfig(
        r=LORA_R,
        lora_alpha=32,
        target_modules=["q_proj", "v_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(base, lora_cfg)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,}  ({100*trainable/total:.3f}%)")

# Verify disable_adapter() works (needed for KL reference)
test_input = tokenizer("test", return_tensors="pt").to(model.device)
with torch.no_grad():
    logits_policy = model(**test_input).logits
    with model.disable_adapter():
        logits_ref = model(**test_input).logits
diff = (logits_policy - logits_ref).abs().max().item()
print(f"Adapter on/off logit-diff: {diff:.4f}  (>0 means adapter is active)")
print("disable_adapter() ✓ working.")


## Step 2: Helpers — observation builder, parser, sampler, log-prob computer

These functions are the contract between the SFT model and the env. The observation JSON schema and the action JSON schema must match `training/generate_sft_data.py` exactly, or the model produces garbage.

In [ ]:
import json, math, random
import numpy as np
import torch
import torch.nn.functional as F
from agentgrid_spaces.runner import HeadlessRunner
from agentgrid_env.server.agentgrid_environment import _spawn_task

# Action vocabulary (SFT was trained on these exact strings)
ACTION_NAMES = ("broadcast", "make_offer", "accept_offer", "execute_task", "renege", "idle")

# Only these kwargs are accepted by the runner for each action — strip everything else
ACTION_KWARGS: dict[str, set[str]] = {
    "broadcast":    {"message"},
    "make_offer":   {"to", "give_type", "give_amount", "want_type", "want_amount"},
    "accept_offer": {"offer_id"},
    "execute_task": set(),
    "renege":       {"offer_id"},
    "idle":         set(),
}


# ── Observation builder ─────────────────────────────────────────────
def build_obs_json(env, agent_id: str) -> dict:
    peers = [p for p in AGENTS if p != agent_id]
    task = env._tasks[agent_id]
    trust_snap = env._trust[agent_id].snapshot_for_obs()

    inbox = [{"from": m["from"], "message": m["message"]}
             for m in env._message_bus if m["from"] != agent_id]

    pending_offers = [
        {
            "offer_id": oid,
            "from": o["from"],
            "to": o["to"],
            "give_type": o["give_type"],
            "give_amount": o["give_amount"],
            "want_type": o["want_type"],
            "want_amount": o["want_amount"],
        }
        for oid, o in env._offers.items()
        if o["to"] == agent_id and o["status"] == "pending"
    ]

    trust_block = {
        p: {
            "Q_accept": round(float(trust_snap.get(f"Q_accept_{p}", 0.0)), 2),
            "Q_trust":  round(float(trust_snap.get(f"Q_trust_{p}",  0.0)), 2),
            "UCB":      round(float(trust_snap.get(f"UCB_{p}",      9.99)), 2),
            "N":        int(trust_snap.get(f"N_{p}", 0)),
        }
        for p in peers
    }

    return {
        "YOUR_AGENT": agent_id,
        "STEP": env._game_step,
        "BATTERY": round(float(env._batteries[agent_id]), 2),
        "TASK": {"energy_cost": task["energy_cost"], "reward_if_done": task["reward_if_done"]},
        "PEERS": {p: {"battery": round(float(env._batteries[p]), 2),
                      "reputation": round(float(env._reputation[p]), 2)} for p in peers},
        "TRUST_MODEL": trust_block,
        "INBOX": inbox,
        "PENDING_OFFERS": pending_offers,
    }


# ── Prompt builder ──────────────────────────────────────────────────
def pick_hint(obs: dict) -> str:
    has_offer = len(obs["PENDING_OFFERS"]) > 0
    low_battery = obs["BATTERY"] < 0.4
    pool = ["execute_task", "idle", "broadcast"]
    if has_offer:
        pool += ["accept_offer", "accept_offer"]
    if low_battery or random.random() < 0.5:
        pool += ["make_offer"]
    return random.choice(pool)


def build_prompt(obs: dict, hint: str) -> str:
    return (
        f"<s>[INST] {json.dumps(obs, indent=2)}\n\n"
        f"Hint: a reasonable next action right now is `{hint}`.\n"
        f"Reply with the JSON action. [/INST]"
    )


# ── Action parser ───────────────────────────────────────────────────
def parse_action(text: str, agent_id: str) -> tuple[str, dict]:
    s = text.strip()
    if s.startswith("```"):
        s = s.split("```", 2)[1]
        if s.startswith("json"):
            s = s[4:]
        s = s.rsplit("```", 1)[0]
    a, b = s.find("{"), s.rfind("}")
    if a == -1 or b == -1 or b <= a:
        return "idle", {}
    try:
        obj = json.loads(s[a:b + 1])
    except json.JSONDecodeError:
        return "idle", {}
    if not isinstance(obj, dict):
        return "idle", {}
    name = obj.get("action")
    if name not in ACTION_NAMES:
        return "idle", {}
    # Only pass kwargs the runner actually accepts for this action
    allowed = ACTION_KWARGS[name]
    kwargs = {k: v for k, v in obj.items() if k in allowed}
    return name, kwargs


# ── Tokenization & sampling ─────────────────────────────────────────
def tokenize_prompt(prompt: str) -> torch.Tensor:
    ids = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).input_ids
    return ids.to(model.device)


@torch.no_grad()
def sample_with_logp(prompt_ids: torch.Tensor, temperature: float
                     ) -> tuple[str, torch.Tensor, torch.Tensor]:
    out = model.generate(
        prompt_ids,
        attention_mask=torch.ones_like(prompt_ids),
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=True,
        temperature=temperature,
        top_p=0.95,
        pad_token_id=tokenizer.eos_token_id,
        return_dict_in_generate=True,
        output_scores=True,
    )
    seq = out.sequences
    completion_ids = seq[:, prompt_ids.shape[1]:]

    if len(out.scores) == 0:
        return "", completion_ids, torch.zeros(0, device=model.device)
    scores = torch.stack(out.scores, dim=1)
    logp = F.log_softmax(scores.float() / max(temperature, 1e-6), dim=-1)
    chosen = completion_ids[:, :scores.shape[1]].unsqueeze(-1)
    logp_old = logp.gather(-1, chosen).squeeze(-1).squeeze(0)

    completion_text = tokenizer.decode(completion_ids[0], skip_special_tokens=True)
    return completion_text, completion_ids, logp_old.detach()


# ── Log-prob recompute ──────────────────────────────────────────────
def compute_logp_and_entropy(prompt_ids: torch.Tensor, completion_ids: torch.Tensor,
                              use_adapter: bool = True
                              ) -> tuple[torch.Tensor, torch.Tensor]:
    full_ids = torch.cat([prompt_ids, completion_ids], dim=1)
    P = prompt_ids.shape[1]
    T = completion_ids.shape[1]

    def _forward():
        out = model(full_ids, attention_mask=torch.ones_like(full_ids))
        logits = out.logits[0]
        logits = logits[P - 1: P - 1 + T]
        logp_full = F.log_softmax(logits.float(), dim=-1)
        chosen = completion_ids[0]
        logp = logp_full.gather(-1, chosen.unsqueeze(-1)).squeeze(-1)
        probs = logp_full.exp()
        entropy = -(probs * logp_full).sum(dim=-1)
        return logp, entropy

    if use_adapter:
        return _forward()
    else:
        with model.disable_adapter():
            with torch.no_grad():
                return _forward()


# ── KL divergence ──────────────────────────────────────────────────
def kl_k3(logp_cur: torch.Tensor, logp_ref: torch.Tensor) -> torch.Tensor:
    r = logp_ref - logp_cur
    return torch.exp(r) - r - 1.0


print("Helpers defined.")

### Smoke test: 1 episode, dump 5 samples

Confirms the format pipeline works **before** burning A10G time on training. If samples show all `idle` or all parse-fails, fix it here — not 30 minutes into the run.

In [ ]:
model.eval()
runner = HeadlessRunner(episode_steps=EPISODE_STEPS)
snap = runner.reset(seed=0)

action_counts = {n: 0 for n in ACTION_NAMES}
parse_fails = 0
sample_lines = []
N_SAMPLES_TO_LOG = 5

step = 0
while not snap.done:
    for agent_id in AGENTS:
        obs = build_obs_json(runner._env, agent_id)
        hint = pick_hint(obs)
        prompt = build_prompt(obs, hint)
        prompt_ids = tokenize_prompt(prompt)
        completion, c_ids, _ = sample_with_logp(prompt_ids, temperature=TEMP_START)
        action, kwargs = parse_action(completion, agent_id)
        action_counts[action] += 1
        if action == "idle" and "idle" not in completion.lower():
            parse_fails += 1
        runner.apply(agent_id, action, **kwargs)
        if len(sample_lines) < N_SAMPLES_TO_LOG:
            sample_lines.append(
                f"--- agent={agent_id} step={step} hint={hint} ---\n"
                f"PROMPT (last 400 chars): ...{prompt[-400:]}\n"
                f"COMPLETION: {completion}\n"
                f"PARSED: action={action} kwargs={kwargs}\n"
            )
    snap = runner.snapshot()
    step += 1

runner._env._dump_trust_decisions()

print("Smoke-test action distribution:", action_counts)
print(f"Likely parse-failures (idle output without 'idle' in text): {parse_fails}")
print(f"Episode return: {sum(snap.rewards.values())/3:.2f}  promise_keep: {snap.promise_keep_ratio:.2f}")
print()
for s in sample_lines:
    print(s)

non_idle_actions = sum(v for k, v in action_counts.items() if k != "idle")
if non_idle_actions < 5:
    print(f"WARNING: only {non_idle_actions} non-idle actions — parser/format degraded. "
          f"Training will still proceed but rewards may be poor until the model warms up.")
else:
    print(f"✓ Smoke test passed — {non_idle_actions} non-idle actions observed.")


## Step 3: GRPO training loop

For each of `N_GROUPS` groups:
1. Roll out `G=4` trajectories with the **same seed** (different sub-seeds for stochasticity), capturing per-token `logp_old` during sampling.
2. Compute group-relative advantages `A_i = (R_i − mean) / (std + ε)`.
3. Re-run forward with grad enabled to get `logp_cur` and `entropy`; query frozen reference via `disable_adapter()` to get `logp_ref`.
4. Loss = PPO-clipped surrogate + β·KL(π‖π_ref) − α·H(π), masked to completion tokens.
5. Backprop, grad-clip, optimizer step.
6. Append per-episode reward log (locked schema) and per-group metrics; checkpoint adapters every 5 groups.

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR

trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = AdamW(trainable_params, lr=GRPO_LR, betas=(0.9, 0.95), weight_decay=0.0)

warmup_groups = max(1, int(WARMUP_FRAC * N_GROUPS))
def lr_lambda(group_idx: int) -> float:
    if group_idx < warmup_groups:
        return float(group_idx + 1) / float(warmup_groups)
    progress = (group_idx - warmup_groups) / max(1, N_GROUPS - warmup_groups)
    return 0.5 * (1.0 + math.cos(math.pi * progress))
scheduler = LambdaLR(optimizer, lr_lambda)

runner = HeadlessRunner(episode_steps=EPISODE_STEPS)
t_start = time.time()
ep_global = 0

def temp_for_group(g_idx: int) -> float:
    frac = g_idx / max(1, N_GROUPS - 1)
    return TEMP_START + (TEMP_END - TEMP_START) * frac


for group_idx in range(N_GROUPS):
    temperature = temp_for_group(group_idx)
    seed_base = group_idx * 1000

    # ── 1. Rollout phase ────────────────────────────────────────────
    model.eval()
    trajectories: list[dict] = []

    for g in range(G):
        snap = runner.reset(seed=seed_base + g)
        scaled = int(ep_global * CURRICULUM_SCALE)
        runner._env._total_episodes_completed = scaled
        runner._env._tasks = {a: _spawn_task(runner._env._rng, scaled) for a in AGENTS}

        traj_steps: list[dict] = []
        while not snap.done:
            for agent_id in AGENTS:
                obs = build_obs_json(runner._env, agent_id)
                hint = pick_hint(obs)
                prompt = build_prompt(obs, hint)
                p_ids = tokenize_prompt(prompt)
                if p_ids.shape[1] > MAX_SEQ_LEN - MAX_NEW_TOKENS:
                    p_ids = p_ids[:, -(MAX_SEQ_LEN - MAX_NEW_TOKENS):]
                completion, c_ids, logp_old = sample_with_logp(p_ids, temperature)
                action, kwargs = parse_action(completion, agent_id)
                runner.apply(agent_id, action, **kwargs)
                traj_steps.append({
                    "p_ids":    p_ids.detach().cpu(),
                    "c_ids":    c_ids.detach().cpu(),
                    "logp_old": logp_old.detach().cpu(),
                })
            snap = runner.snapshot()

        runner._env._dump_trust_decisions()

        episode_return = sum(snap.rewards.values()) / len(AGENTS)
        trajectories.append({
            "steps":        traj_steps,
            "return":       float(episode_return),
            "rewards":      dict(snap.rewards),
            "promise_keep": float(snap.promise_keep_ratio),
        })

        with open(LOG_PATH, "a") as f:
            f.write(json.dumps({
                "rewards":      snap.rewards,
                "promise_keep": snap.promise_keep_ratio,
            }) + "\n")
        ep_global += 1

    # ── 2. Group-relative advantages ────────────────────────────────
    returns = np.array([t["return"] for t in trajectories], dtype=np.float64)
    adv_raw = returns - returns.mean()
    adv_std = returns.std()
    advantages = adv_raw / (adv_std + 1e-8)

    # ── 3. Gradient phase ──────────────────────────────────────────
    model.train()
    optimizer.zero_grad(set_to_none=True)

    sum_policy_loss = 0.0
    sum_kl          = 0.0
    sum_entropy     = 0.0
    sum_clipfrac    = 0.0
    n_steps         = 0

    for traj, adv in zip(trajectories, advantages):
        adv_t = torch.tensor(float(adv), device=model.device, dtype=torch.float32)
        for step in traj["steps"]:
            p_ids    = step["p_ids"].to(model.device)
            c_ids    = step["c_ids"].to(model.device)
            logp_old = step["logp_old"].to(model.device, dtype=torch.float32)
            T_old    = logp_old.shape[0]
            if T_old == 0 or c_ids.shape[1] == 0:
                continue

            logp_cur, ent_cur = compute_logp_and_entropy(p_ids, c_ids, use_adapter=True)
            logp_ref, _       = compute_logp_and_entropy(p_ids, c_ids, use_adapter=False)

            T = min(logp_cur.shape[0], logp_ref.shape[0], T_old, c_ids.shape[1])
            if T == 0:
                continue
            logp_cur = logp_cur[:T]
            logp_ref = logp_ref[:T]
            logp_old = logp_old[:T]
            ent_cur  = ent_cur[:T]

            ratio  = torch.exp(logp_cur - logp_old)
            surr1  = ratio * adv_t
            surr2  = torch.clamp(ratio, 1.0 - CLIP_EPS, 1.0 + CLIP_EPS) * adv_t
            policy_loss = -torch.min(surr1, surr2).mean()

            kl      = kl_k3(logp_cur, logp_ref).mean()
            entropy = ent_cur.mean()

            # Accumulate raw (un-normalized) loss; normalize by n_steps after the loop
            loss = policy_loss + KL_BETA * kl - ENT_ALPHA * entropy
            loss.backward()

            with torch.no_grad():
                clipped = ((ratio < 1.0 - CLIP_EPS) | (ratio > 1.0 + CLIP_EPS)).float().mean().item()
            sum_policy_loss += policy_loss.item()
            sum_kl          += kl.item()
            sum_entropy     += entropy.item()
            sum_clipfrac    += clipped
            n_steps         += 1

    # Normalize accumulated gradients by actual number of gradient steps
    if n_steps > 0:
        for p in trainable_params:
            if p.grad is not None:
                p.grad.div_(n_steps)

    grad_norm = torch.nn.utils.clip_grad_norm_(trainable_params, GRAD_CLIP).item()
    optimizer.step()
    scheduler.step()
    optimizer.zero_grad(set_to_none=True)

    # ── 4. Logging ─────────────────────────────────────────────────
    avg_policy_loss = sum_policy_loss / max(1, n_steps)
    avg_kl          = sum_kl          / max(1, n_steps)
    avg_entropy     = sum_entropy     / max(1, n_steps)
    avg_clipfrac    = sum_clipfrac    / max(1, n_steps)
    cur_lr          = scheduler.get_last_lr()[0]
    elapsed_min     = (time.time() - t_start) / 60

    with open(METRICS_PATH, "a") as f:
        f.write(json.dumps({
            "group":              group_idx,
            "ep_global":          ep_global,
            "policy_loss":        avg_policy_loss,
            "kl":                 avg_kl,
            "entropy":            avg_entropy,
            "clip_frac":          avg_clipfrac,
            "grad_norm":          grad_norm,
            "lr":                 cur_lr,
            "group_mean_return":  float(returns.mean()),
            "group_std_return":   float(returns.std()),
            "temperature":        temperature,
            "elapsed_min":        elapsed_min,
            "n_steps":            n_steps,
        }) + "\n")

    print(
        f"[g={group_idx:3d}/{N_GROUPS}] "
        f"R={returns.mean():+.2f}±{returns.std():.2f}  "
        f"PL={avg_policy_loss:+.4f}  KL={avg_kl:.4f}  H={avg_entropy:.3f}  "
        f"clip={avg_clipfrac:.2f}  gn={grad_norm:.2f}  lr={cur_lr:.2e}  "
        f"t={elapsed_min:.1f}m  steps={n_steps}"
    )

    if avg_kl > 1.0:
        print(f"  ⚠ KL={avg_kl:.2f} exceeded 1.0 — policy diverging. Lower lr or raise KL_BETA.")
    if math.isnan(avg_policy_loss):
        print("  ⚠ Loss is NaN — stopping.")
        break

    if (group_idx + 1) % CKPT_EVERY == 0:
        ckpt_path = os.path.join(CKPT_DIR, f"group_{group_idx + 1:03d}")
        model.save_pretrained(ckpt_path)
        print(f"  ↳ adapter checkpoint: {ckpt_path}")

    del trajectories
    torch.cuda.empty_cache()

print(f"\nGRPO training complete in {(time.time() - t_start) / 60:.1f} min "
      f"({ep_global} episodes across {N_GROUPS} groups).")


## Step 4: Plots — reward curve + training health

Two outputs:
- **`grpo_curve.png`** — episode return + smoothed mean (the "did training work" plot)
- **`grpo_training_health.png`** — KL, entropy, clip fraction, group-mean return (the "is training stable" diagnostic)

Both consume the per-group metrics file `grpo_metrics.jsonl` and per-episode `grpo_rewards.jsonl`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Load per-episode rewards
ep_rewards = []
ep_promise = []
with open(LOG_PATH) as f:
    for line in f:
        d = json.loads(line)
        ep_rewards.append(sum(d["rewards"].values()) / 3)
        ep_promise.append(d["promise_keep"])
ep_rewards = np.array(ep_rewards)
ep_promise = np.array(ep_promise)

# Load per-group metrics
metrics = []
with open(METRICS_PATH) as f:
    for line in f:
        metrics.append(json.loads(line))

# ── Plot 1: reward curve ────────────────────────────────────────────
window = max(5, len(ep_rewards) // 20)
smooth = np.convolve(ep_rewards, np.ones(window) / window, mode="valid")

plt.figure(figsize=(10, 5))
plt.plot(ep_rewards, alpha=0.3, label="Episode return")
plt.plot(range(window - 1, len(ep_rewards)), smooth, label=f"Smoothed (w={window})")
plt.axhline(7.59, color="grey", ls="--", alpha=0.5, label="Random baseline (7.59)")
plt.xlabel("Episode")
plt.ylabel("Avg episode return (3 agents)")
plt.title("GRPO Self-Play — AgentGrid Sim")
plt.legend()
plt.tight_layout()
plt.savefig("/tmp/workspace/grpo_curve.png", dpi=150)
plt.show()
print("Saved: /tmp/workspace/grpo_curve.png")

# ── Plot 2: training health (2x2) ───────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
groups = [m["group"] for m in metrics]

axes[0, 0].plot(groups, [m["group_mean_return"] for m in metrics], "o-", label="group mean")
axes[0, 0].fill_between(
    groups,
    [m["group_mean_return"] - m["group_std_return"] for m in metrics],
    [m["group_mean_return"] + m["group_std_return"] for m in metrics],
    alpha=0.2,
)
axes[0, 0].axhline(7.59, color="grey", ls="--", alpha=0.5)
axes[0, 0].set_title("Group-mean return ± std")
axes[0, 0].set_xlabel("Group"); axes[0, 0].set_ylabel("Return")

axes[0, 1].plot(groups, [m["kl"] for m in metrics], "o-")
axes[0, 1].axhline(KL_BETA * 10, color="red", ls=":", alpha=0.5, label="alarm threshold")
axes[0, 1].set_title("KL(π‖π_ref) — should plateau")
axes[0, 1].set_xlabel("Group"); axes[0, 1].set_ylabel("KL")
axes[0, 1].legend()

axes[1, 0].plot(groups, [m["entropy"] for m in metrics], "o-")
axes[1, 0].set_title("Policy entropy — should slowly decrease")
axes[1, 0].set_xlabel("Group"); axes[1, 0].set_ylabel("Entropy (nats)")

axes[1, 1].plot(groups, [m["policy_loss"] for m in metrics], "o-", label="policy loss")
axes[1, 1].plot(groups, [m["clip_frac"] for m in metrics], "s-", label="clip frac")
axes[1, 1].set_title("Policy loss & clip fraction")
axes[1, 1].set_xlabel("Group")
axes[1, 1].legend()

plt.suptitle(f"GRPO Training Health — {N_GROUPS} groups × G={G}")
plt.tight_layout()
plt.savefig("/tmp/workspace/grpo_training_health.png", dpi=150)
plt.show()
print("Saved: /tmp/workspace/grpo_training_health.png")

# ── Pass/fail summary against training_guide.md targets ─────────────
last100 = ep_rewards[-100:] if len(ep_rewards) >= 100 else ep_rewards
last100_pk = ep_promise[-100:] if len(ep_promise) >= 100 else ep_promise
mean_R = float(last100.mean())
mean_PK = float(last100_pk.mean())
print(f"\n── Submission pass/fail ──")
print(f"Last-{len(last100)} mean return  : {mean_R:.2f}  (baseline 7.59) — "
      f"{'PASS ✓' if mean_R > 7.59 else 'FAIL ✗'}")
print(f"Last-{len(last100_pk)} mean promise-keep: {mean_PK:.2f}  (target 0.40) — "
      f"{'PASS ✓' if mean_PK > 0.40 else 'FAIL ✗'}")

## Step 5: Sanity-check artifacts and push checkpoint

Before pushing to Hub, confirm artifact schemas match what the eval scripts expect.

In [ ]:
# ── Artifact sanity checks ────────────────────────────────────────
import os, json

print("Artifacts produced:")
for path in [LOG_PATH, METRICS_PATH, TRUST_DECISIONS_PATH,
             "/tmp/workspace/grpo_curve.png", "/tmp/workspace/grpo_training_health.png"]:
    exists = os.path.exists(path)
    size = os.path.getsize(path) if exists else 0
    print(f"  {'✓' if exists else '✗'} {path}  ({size:,} bytes)")

# Verify grpo_rewards.jsonl schema (consumed by eval/plot_three_curves.py)
with open(LOG_PATH) as f:
    first_line = f.readline()
d = json.loads(first_line)
assert set(d) == {"rewards", "promise_keep"}, f"Wrong keys: {set(d)}"
assert len(d["rewards"]) == 3, f"Expected 3-agent rewards, got {len(d['rewards'])}"
n_eps = sum(1 for _ in open(LOG_PATH))
print(f"\n✓ grpo_rewards.jsonl: {n_eps} episodes, schema OK")

# Verify trust_decisions.jsonl schema (consumed by eval/plot_trust_correlation.py)
if os.path.exists(TRUST_DECISIONS_PATH):
    with open(TRUST_DECISIONS_PATH) as f:
        first = f.readline()
    if first.strip():
        td = json.loads(first)
        required = {"Q_chosen", "Q_alternatives", "step"}
        assert required <= set(td), f"Missing keys: {required - set(td)}"
        n_td = sum(1 for _ in open(TRUST_DECISIONS_PATH))
        print(f"✓ trust_decisions.jsonl: {n_td} events, schema OK")
    else:
        print("⚠ trust_decisions.jsonl is empty — no accept_offer fired during training")
else:
    print("⚠ trust_decisions.jsonl missing")

print("\nDownload these from Colab to local repo (place under eval/plots/):")
print(f"  - {LOG_PATH}              → eval/plots/grpo_rewards.jsonl")
print(f"  - {TRUST_DECISIONS_PATH} → eval/plots/trust_decisions.jsonl")
print(f"  - /tmp/workspace/grpo_curve.png            → eval/plots/grpo_curve.png")
print(f"  - /tmp/workspace/grpo_training_health.png  → eval/plots/grpo_training_health.png")

In [ ]:
# Push GRPO checkpoint to HF Hub
# Use a fresh write-scoped token from https://huggingface.co/settings/tokens
# DO NOT paste the token in the notebook — pass via prompt or env var.
from huggingface_hub import login

# Option A (interactive): prompts for token
login()
# Option B (env var): set HF_TOKEN before launching, then:
# login(token=os.environ["HF_TOKEN"])

model.push_to_hub(GRPO_REPO)
tokenizer.push_to_hub(GRPO_REPO)
print(f"GRPO checkpoint saved to hf.co/{GRPO_REPO}")